# Duplex Tools on Kaggle: CPU preparation and human voice gate

This notebook pulls the adapter package from `smfabrar/minicpm_tool`. It has two phases.

| Phase | Run these cells | Accelerator | Purpose |
|---|---|---|---|
| A | 1–6, ending after the Granite caller probe | **None** | Select a tag, install the package, run controller tests, inspect parser behavior, and characterize the real Granite 350M caller. |
| B | Rerun cells 1–2, then cells 7 onward | **GPU** | Pull the same tag, build the patched MiniCPM runtime, attach or download GGUF modules, and speak to the persistent session. |

Enable **Internet** in Kaggle Settings. The first code cell contains only `SOURCE_REF`; changing that one tag and rerunning cells 1–2 updates the package. Adding a GPU restarts the notebook runtime, so Python objects from phase A disappear. After the restart rerun cells 1–2 and continue at the GPU section. Skip the CPU benchmark on the GPU clock. An attached Kaggle Dataset containing the GGUF folder saves download time; the model download cell is a fallback.

The voice gate is push-to-talk: record a turn, send it, then listen. It tests a real model-selected tool and a spoken answer in one MiniCPM session. It does not yet demonstrate continuous simultaneous recording and playback. The HTTP API reports context submission; actual KV evaluation remains **unknown** without a native acknowledgement. In the GPU phase, Whisper Tiny and Granite 350M use GPU 1 in FP16 while the native MiniCPM runtime can use both GPUs.

Sources: [Kaggle notebooks](https://www.kaggle.com/docs/notebooks), [IBM Granite model card](https://huggingface.co/ibm-granite/granite-4.0-350m), [Granite tool format](https://github.com/ibm-granite/granite-4.0-language-models/blob/main/Granite%204.0%20Prompt%20engineering%20guide%20v2.md), [MiniCPM GGUF modules](https://huggingface.co/openbmb/MiniCPM-o-4_5-gguf).


In [ ]:
# 1 — The only line to edit when selecting a newer tested release.
SOURCE_REF = 'v0.1.17'
print('Selected release:', SOURCE_REF)


In [ ]:
# 2 — Pull and activate SOURCE_REF. Rerun after changing the tag or switching to GPU.
from pathlib import Path
import subprocess, sys

ROOT = Path('/kaggle/working/minicpm_tool')
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', SOURCE_REF,
                    'https://github.com/smfabrar/minicpm_tool.git', str(ROOT)], check=True)
else:
    current = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip()
    tagged = subprocess.run(['git', '-C', str(ROOT), 'rev-parse', '-q', '--verify', SOURCE_REF],
                            capture_output=True, text=True)
    if tagged.returncode != 0 or tagged.stdout.strip() != current:
        subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin', 'tag', SOURCE_REF], check=True)
        subprocess.run(['git', '-C', str(ROOT), 'checkout', SOURCE_REF], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(ROOT)], check=True)
# Editable installs add a .pth file for the next interpreter start. Make the
# package visible in this already-running notebook kernel immediately.
source_path = str(ROOT / 'src')
if source_path not in sys.path:
    sys.path.insert(0, source_path)
import importlib
importlib.invalidate_caches()
# Remove modules imported from an older selected tag in this kernel.
for module_name in list(sys.modules):
    if module_name == 'duplex_tools' or module_name.startswith('duplex_tools.'):
        del sys.modules[module_name]
import duplex_tools
assert Path(duplex_tools.__file__).resolve().is_relative_to(ROOT.resolve())
print('Package:', ROOT)
print('Imported:', duplex_tools.__file__)
print('Revision:', subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
# 3 — CPU only: check controller, correction, cancellation, parser, and adapter behavior.
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(ROOT / 'tests'), '-v'], check=True)


In [ ]:
# 4 — CPU only: inspect the native Granite parser and simulated adapter.
import asyncio
from duplex_tools.caller import parse_granite_output
from duplex_tools.simulated import SimulatedAdapter
from duplex_tools.contracts import AdapterCapabilities

sample = '<tool_call>{"name":"room_lookup","arguments":{"name":"robotics seminar"}}</tool_call>'
print(parse_granite_output(sample))
sim = SimulatedAdapter(AdapterCapabilities(True, False, False, False, True))
print('Simulated capability profile:', sim.capabilities())


In [ ]:
# 5 — CPU only: load Granite once and measure staged model routing and arguments.
# Kaggle normally has PyTorch. This installs only the model-side packages.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.54,<5', 'accelerate'], check=True)
import json, time
import importlib
from datetime import datetime, timezone
import duplex_tools.caller as caller_module
importlib.reload(caller_module)
from duplex_tools.caller import GraniteToolCaller, TransformersGraniteGenerator
from duplex_tools.contracts import TranscriptSegment
from duplex_tools.tools import (TOOL_SCHEMAS, normalize_calculator_expression,
                                validate_call, validate_grounding)

if 'generator' not in globals():
    generator = TransformersGraniteGenerator(device='cpu')
caller = GraniteToolCaller(generator)
cases = json.loads((ROOT / 'fixtures' / 'caller_cases.json').read_text())
rows = []
for case in cases:
    segment = TranscriptSegment(case['id'], 1, datetime.now(timezone.utc), case['text'], True)
    start = time.perf_counter()
    action = await caller.decide(segment, {}, TOOL_SCHEMAS)
    expected_tool = case.get('tool')
    expected_args = case.get('arguments', {})
    try:
        parsed_args = validate_call(action.tool or '', action.arguments) if action.kind == 'call' else {}
    except ValueError:
        parsed_args = {}
    canonical_args = dict(parsed_args)
    grounded = True
    if action.kind == 'call':
        try:
            validate_grounding(action.tool or '', parsed_args, case['text'])
            if action.tool == 'calculator':
                canonical_args['expression'] = normalize_calculator_expression(parsed_args['expression'])
        except (KeyError, ValueError):
            grounded = False
    args_match = parsed_args == expected_args or (
        case['id'] == 'calculator' and parsed_args.get('expression', '').replace(' ', '') == expected_args.get('expression', '').replace(' ', '')
    )
    canonical_match = canonical_args == expected_args or (
        case['id'] == 'calculator' and canonical_args.get('expression', '').replace(' ', '') == expected_args.get('expression', '').replace(' ', '')
    )
    route_pass = action.kind == case['kind'] and (expected_tool is None or action.tool == expected_tool)
    passed = route_pass and (expected_tool is None or args_match)
    rows.append({'id': case['id'], 'expected': case['kind'], 'actual': action.kind,
                 'tool': action.tool, 'arguments': dict(action.arguments),
                 'canonical_arguments': canonical_args, 'route_pass': route_pass,
                 'raw_arguments_pass': None if expected_tool is None else args_match,
                 'canonical_arguments_pass': None if expected_tool is None else canonical_match,
                 'grounded': grounded, 'passed': passed,
                 'latency_s': round(time.perf_counter() - start, 2), 'raw': action.raw})
for row in rows:
    print(json.dumps(row, ensure_ascii=False))
valid_calls_executable = all(
    r['route_pass'] and r['canonical_arguments_pass'] and r['grounded']
    for r in rows if r['id'] in {'room', 'calculator', 'document'}
)
unsafe_proposals_blocked = all(
    r['actual'] != 'call' or not r['grounded']
    for r in rows if r['id'] in {'none', 'incomplete', 'clarify'}
)
gpu_ready = valid_calls_executable and unsafe_proposals_blocked
print('Raw caller result:', sum(r['passed'] for r in rows), '/', len(rows),
      '(retain this as caller evidence)')
print('Valid calls executable after declared normalization:', valid_calls_executable)
print('Unsafe proposals blocked before execution:', unsafe_proposals_blocked)
print('GPU experiment readiness:',
      'PASS — proceed with the known caller limitations' if gpu_ready else
      'FAIL — a valid call cannot execute or an unsafe proposal can reach a tool')


In [ ]:
# 6 — CPU only: edit this sentence for a quick human-written transcript probe.
sentence = 'Where is the robotics seminar?'
segment = TranscriptSegment('manual-text', 1, datetime.now(timezone.utc), sentence, True)
proposal = await caller.decide(segment, {}, TOOL_SCHEMAS)
print('Proposal:', proposal)
print('Raw model output:', proposal.raw)


## CPU work is complete; enable the GPU

Proceed when **GPU experiment readiness** reports PASS. Perfect raw caller accuracy is not an admission condition: raw mistakes remain experimental results, while the admission check asks whether intended calls are executable and unintended proposals are stopped before execution. In Kaggle Settings select **Accelerator → GPU**. Kaggle restarts the runtime. Rerun **cells 1–2**, then continue below. Phase B needs CUDA for MiniCPM. The 350M caller and tiny speech recognizer use the second GPU in FP16 during the human experiment. If the GPU has too little memory, reduce `N_GPU_LAYERS` in the server cell and record the value and resulting latency.


In [ ]:
# 7 — GPU phase: verify the accelerator and install only the voice dependencies.
import torch
assert torch.cuda.is_available(), 'Enable a GPU in Kaggle Settings before continuing.'
print(torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers>=4.54,<5', 'accelerate', 'huggingface_hub',
                'gradio>=5,<7', 'soundfile', 'faster-whisper'], check=True)


In [ ]:
# 8 — Prefer an attached Kaggle Dataset with this GGUF folder; otherwise download.
from huggingface_hub import snapshot_download

# Example: Path('/kaggle/input/your-minicpm-gguf/MiniCPM-o-4_5-gguf')
ATTACHED_MODEL_DIR = None
MODEL_DIR = Path(ATTACHED_MODEL_DIR) if ATTACHED_MODEL_DIR else Path('/kaggle/working/models/MiniCPM-o-4_5-gguf')
required = [
    'MiniCPM-o-4_5-Q4_K_M.gguf',
    'audio/MiniCPM-o-4_5-audio-F16.gguf',
    'tts/MiniCPM-o-4_5-tts-F16.gguf',
    'tts/MiniCPM-o-4_5-projector-F16.gguf',
    'token2wav-gguf/encoder.gguf',
    'token2wav-gguf/flow_matching.gguf',
    'token2wav-gguf/flow_extra.gguf',
    'token2wav-gguf/hifigan2.gguf',
    'token2wav-gguf/prompt_cache.gguf',
]
if not all((MODEL_DIR / name).is_file() for name in required):
    assert ATTACHED_MODEL_DIR is None, 'Attached dataset is missing required files.'
    snapshot_download(repo_id='openbmb/MiniCPM-o-4_5-gguf',
                      allow_patterns=required, local_dir=str(MODEL_DIR))
missing = [name for name in required if not (MODEL_DIR / name).is_file()]
assert not missing, f'Missing GGUF modules: {missing}'
print('Model directory:', MODEL_DIR)


In [ ]:
# 9 — Restore a saved T4 runtime, or build the pinned runtime once.
import shutil
from duplex_tools.runtime_bundle import verify_runtime_bundle

# Rerunning this cell intentionally ends the current conversation before
# repairing or replacing its native runtime. Retain files and build objects.
if 'server_process' in globals() and server_process.poll() is None:
    server_process.terminate()
    try:
        server_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        server_process.kill()
        server_process.wait(timeout=10)
    print('Stopped previous MiniCPM server.')
if 'app' in globals():
    app.close()
    print('Closed previous Gradio interface.')

# Future sessions: attach the saved notebook output and set this directory.
# Example: '/kaggle/input/YOUR-NOTEBOOK-OUTPUT/minicpm_omni_runtime_sm75'
ATTACHED_RUNTIME_DIR = None
UPSTREAM = Path('/kaggle/working/llama.cpp-omni')
PIN = '64d092c60db4b4ee45768476bd752f03fdcc98ea'
patch = ROOT / 'fixtures' / 'context-injection.patch'
if ATTACHED_RUNTIME_DIR:
    RUNTIME_BUNDLE = Path('/kaggle/working/minicpm_omni_runtime_sm75')
    if RUNTIME_BUNDLE.exists():
        import shutil
        shutil.rmtree(RUNTIME_BUNDLE)
    shutil.copytree(Path(ATTACHED_RUNTIME_DIR), RUNTIME_BUNDLE, symlinks=True)
    SERVER_BIN = verify_runtime_bundle(RUNTIME_BUNDLE, expected_pin=PIN, expected_patch=patch)
    print('Restored verified runtime; compilation skipped:', SERVER_BIN)
else:
    if not UPSTREAM.exists():
        subprocess.run(['git', 'clone', 'https://github.com/tc-mb/llama.cpp-omni.git', str(UPSTREAM)], check=True)
    subprocess.run(['git', '-C', str(UPSTREAM), 'checkout', PIN], check=True)
    # Upgrade an existing extraction patch without discarding compiled objects.
    source = (UPSTREAM / 'tools/server/server-omni.cpp').read_text()
    if '"next_cnt", 1' in source:
        subprocess.run([sys.executable, str(ROOT / 'scripts/repair_runtime.py'),
                        '--upstream', str(UPSTREAM), '--patch-only'], check=True)
    reverse = subprocess.run(['git', '-C', str(UPSTREAM), 'apply', '--reverse', '--check', str(patch)], capture_output=True)
    if reverse.returncode != 0:
        subprocess.run(['git', '-C', str(UPSTREAM), 'apply', '--check', str(patch)], check=True)
        subprocess.run(['git', '-C', str(UPSTREAM), 'apply', str(patch)], check=True)
    # Kaggle exposes the CUDA runtime and cuBLAS but may omit the unversioned
    # libcuda.so needed for CUDA VMM. CUDA kernels and offload remain enabled.
    subprocess.run(['cmake', '-S', str(UPSTREAM), '-B', str(UPSTREAM / 'build'),
                    '-DCMAKE_BUILD_TYPE=Release', '-DGGML_CUDA=ON',
                    '-DGGML_CUDA_NO_VMM=ON', '-DLLAMA_OPENSSL=OFF'], check=True)
    subprocess.run(['cmake', '--build', str(UPSTREAM / 'build'),
                    '--target', 'llama-omni-server', '-j', '2'], check=True)
    SERVER_BIN = UPSTREAM / 'build' / 'bin' / 'llama-omni-server'
    assert SERVER_BIN.is_file()


In [ ]:
# 10 — Package a new build with provenance and dependency checks.
# Run this after the build above. It is quick when a saved runtime was restored.
from duplex_tools.runtime_bundle import create_runtime_bundle, runtime_environment

if not ATTACHED_RUNTIME_DIR:
    RUNTIME_BUNDLE = Path('/kaggle/working/minicpm_omni_runtime_sm75')
    SERVER_BIN = create_runtime_bundle(
        UPSTREAM / 'build' / 'bin', RUNTIME_BUNDLE,
        patch=patch, source_pin=PIN, source_ref=SOURCE_REF,
        build_options=['GGML_CUDA=ON', 'GGML_CUDA_NO_VMM=ON',
                       'LLAMA_OPENSSL=OFF', 'CMAKE_BUILD_TYPE=Release', 'CUDA_ARCH=75-real'],
    )
    print('Reusable runtime created:', RUNTIME_BUNDLE)
    print('Size:', subprocess.check_output(['du', '-sh', str(RUNTIME_BUNDLE)], text=True).split()[0])
    print('After the human experiment, Quick Save this notebook with output files.')
else:
    print('Using attached verified runtime:', RUNTIME_BUNDLE)
SERVER_ENV = runtime_environment(RUNTIME_BUNDLE)


In [ ]:
# 11 — Start one persistent local MiniCPM server and wait for HTTP readiness.
import socket, subprocess, time, urllib.request

N_GPU_LAYERS = 99
def free_local_port(candidates=(19080, 18080, 9060, 8765, 49152)):
    for candidate in candidates:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:
            probe.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            try:
                probe.bind(('127.0.0.1', candidate))
            except OSError:
                continue
            return candidate
    raise RuntimeError('No candidate localhost port is available')

SERVER_LOG = Path('/kaggle/working/minicpm_server.log')
if 'server_process' not in globals() or server_process.poll() is not None:
    SERVER_PORT = free_local_port()
    BASE_URL = f'http://127.0.0.1:{SERVER_PORT}'
    server_log_handle = SERVER_LOG.open('w')
    server_process = subprocess.Popen([
        str(SERVER_BIN), '--host', '127.0.0.1', '--port', str(SERVER_PORT),
        '--model', str(MODEL_DIR / 'MiniCPM-o-4_5-Q4_K_M.gguf'),
        '-ngl', str(N_GPU_LAYERS), '--ctx-size', '8192',
    ], stdout=server_log_handle, stderr=subprocess.STDOUT, env=SERVER_ENV,
       start_new_session=True)
for _ in range(120):
    if server_process.poll() is not None:
        raise RuntimeError(SERVER_LOG.read_text()[-4000:])
    try:
        with urllib.request.urlopen(BASE_URL + '/health', timeout=2) as response:
            if response.status == 200:
                break
    except Exception:
        time.sleep(2)
else:
    raise TimeoutError('MiniCPM server did not become ready; inspect ' + str(SERVER_LOG))
print('MiniCPM server is ready at', BASE_URL)


In [ ]:
# 12 — Load CPU caller and recognizer once, then initialize MiniCPM once.
from faster_whisper import WhisperModel
from duplex_tools.caller import GraniteToolCaller, TransformersGraniteGenerator
from duplex_tools.controller import ContextController, JsonlEventLog
from duplex_tools.conversation import ConversationRouter
from duplex_tools.minicpm_client import MiniCPMStreamSession, OmniHttpClient
from duplex_tools.tools import RoomLookup, SafeCalculator, DocumentSearch
from duplex_tools.voice_demo import VoiceDemo, make_gradio_ui

OUTPUT = Path('/kaggle/working/duplex_voice_output')
OUTPUT.mkdir(exist_ok=True)
tools = {
    'room_lookup': RoomLookup({'robotics seminar': 'The robotics seminar is in room B742.',
                               'vision seminar': 'The vision seminar is in room C314.'}),
    'calculator': SafeCalculator(),
    'document_search': DocumentSearch({'Thesis deadlines': 'The draft is due on October 15; the final copy is due on November 20.',
                                       'Lab access': 'The lab is open Monday through Friday from 9 to 17.'}),
}
controller = ContextController(tools, log=JsonlEventLog(OUTPUT / 'controller.jsonl'))
AUX_GPU = 1 if torch.cuda.device_count() > 1 else 0
AUX_DEVICE = f'cuda:{AUX_GPU}'
router = ConversationRouter(GraniteToolCaller(TransformersGraniteGenerator(device=AUX_DEVICE)), controller)
recognizer = WhisperModel('tiny.en', device='cuda', device_index=AUX_GPU, compute_type='float16')
session = MiniCPMStreamSession(OmniHttpClient(BASE_URL, timeout_s=600), controller)
print(await session.initialize(output_dir=str(OUTPUT), model_dir=str(MODEL_DIR),
                               tts_bin_dir=str(MODEL_DIR / 'tts'), token2wav_device='gpu:0'))
voice_demo = VoiceDemo(session, router, OUTPUT, recognizer,
                       transcription_backend=f'Whisper Tiny FP16 on {AUX_DEVICE}',
                       routing_backend=f'Granite 350M FP16 on {AUX_DEVICE}')
print('Adapter capabilities:', session.capabilities())
print('Auxiliary inference device:', AUX_DEVICE)


In [ ]:
# 13 — Human voice gate. The Gradio share URL is public; this one has a random password.
import secrets
password = secrets.token_urlsafe(12)
app = make_gradio_ui(voice_demo)
app.launch(share=True, auth=('tester', password), inline=False, prevent_thread_lock=True)
print('Gradio username: tester')
print('Gradio password:', password)
print('Send returns immediately; the page polls the persistent background turn once per second.')
print('Recovery snapshot:', OUTPUT / 'latest_turn.json')


## Human test procedure

1. In the Gradio page, record **“Where is the robotics seminar?”** Send it. Check the user transcript, that the proposal names `room_lookup`, the submitted event in the trace, and listen for **B742**.
2. Ask **“What is 17 times 23?”** Listen for **391**. Inspect the exact expression chosen; an incorrect argument is a caller failure even if the speech sounds plausible.
3. Ask for the **thesis deadline**, then try an ordinary greeting. The greeting should make no tool call.
4. Repeat with your own paraphrases. Save a verdict after listening to every answer. Logs are in `/kaggle/working/duplex_voice_output/` as `human_trials.jsonl`, `human_verdicts.jsonl`, and `controller.jsonl`.

`Send turn` starts a persistent background job and returns immediately. The page polls it once per second with short non-queued requests, so the experiment can continue if the public Gradio connection briefly drops. `latest_turn.json` is rewritten after each stage and preserves the current transcript, status, error, and final result. If the page reconnects, the poller restores the latest result. The Granite router has a 90-second deadline; crossing it records a failed trial rather than leaving an unbounded spinner. The stage text records the GPU used by Whisper and Granite.

The model input counter and session stay live across all turns. `evaluation: unknown` means the HTTP prefill accepted the context but did not confirm token evaluation. Generated text and the audio file are recorded separately; the listening verdict is the spoken-answer ground truth. A missing or incorrect answer is a failed trial, not something to infer away from the tool trace.

The microphone gate records complete turns. Correction or cancellation during an actively running tool, overlapping speech, and live incremental playback remain the next human gates after this first extraction.

## Preserve the compiled runtime

After completing the human trials, choose **Save Version → Quick Save** and include the current output files. Kaggle preserves files under `/kaggle/working` up to its output limit. In a later notebook, use **Add Input**, attach this notebook's saved output, and set `ATTACHED_RUNTIME_DIR` in cell 9 to the attached `minicpm_omni_runtime_sm75` directory. Cell 9 verifies the source commit, patch hash, every bundled file, the T4 `sm_75` architecture, and dynamic system libraries before skipping compilation.

The bundle intentionally excludes model weights and Kaggle system libraries. Keep model weights in their own Kaggle Dataset. CUDA, glibc, OpenSSL, and other host libraries are recorded in `manifest.json` and checked by `ldd` during restoration. This artifact must be used on a T4; a P100 or other architecture requires its own build.
